# Demo G — Pretraining on unlabelled data, then a few labels

**Class 11 · G1–G2** (slides: `slides/11_autoencoders.md`)

* **G1:** pretrain an encoder on the 1257 unlabelled training digits (reconstruction), then classify with 1 to 20 labelled digits per class: raw pixels, PCA codes, autoencoder codes (frozen), the encoder fine-tuned, and the same network trained from scratch. 3 draws of the labels per size.
* **G2:** the **pretext task**: reconstruct from a masked input (a fraction of the pixels hidden). Sweep of the masking rate and a failure case: an encoder pretrained on noise.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import numpy as np
import polars as pl
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from faa import demo, plotlib, semisup
from faa.demo import show, slide

demo.setup("11")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
X, y = semisup.digits_flat()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
K = 16


def lr_acc(Ztr, Zte, idx):
    return LogisticRegression(C=10, max_iter=3000).fit(Ztr[idx], ytr[idx]).score(Zte, yte)

## Slide: Pretrain, Then Label (G1)

Encoder $64 \to 32 \to 16$ (ReLU), decoder mirrored; pretraining on all 1257 training digits **without labels** (150 epochs, a denoising pretext with noise 0.1). Fine-tuning: encoder plus a softmax layer, 100 epochs on the labelled digits.
One pretrained encoder is shared by the 3 draws. The scratch network has the same architecture, random start.

In [ ]:
ae, enc = semisup.autoencoder(64, K, hidden=(32,), seed=0)
semisup.fit_ae(ae, Xtr, noise=0.1, epochs=150)
pca = PCA(K).fit(Xtr)
Ztr_ae, Zte_ae = np.asarray(enc.predict(Xtr, verbose=0)), np.asarray(enc.predict(Xte, verbose=0))
_, enc_fresh = semisup.autoencoder(64, K, hidden=(32,), seed=0)
rows = []
for per in (1, 2, 5, 10, 20):
    r = {"raw": [], "pca": [], "ae_frozen": [], "ae_finetuned": [], "scratch": []}
    for seed in range(3):
        idx = semisup.few_labels(ytr, per, seed)
        r["raw"].append(lr_acc(Xtr, Xte, idx))
        r["pca"].append(lr_acc(pca.transform(Xtr), pca.transform(Xte), idx))
        r["ae_frozen"].append(lr_acc(Ztr_ae, Zte_ae, idx))
        for name, base in (("ae_finetuned", enc), ("scratch", enc_fresh)):
            m = semisup.classifier_from(base, seed=seed)
            m.fit(Xtr[idx], ytr[idx], epochs=100, batch_size=16, verbose=0)
            r[name].append(float(m.evaluate(Xte, yte, verbose=0)[1]))
    rows.append({"per_class": per, **{k: float(np.mean(v)) for k, v in r.items()}})
res = pl.DataFrame(rows)
slide("Pretrain, Then Label")
show(res)

fig, ax = plotlib.fig("col", 1.9)
for col, c, mk in (
    ("raw", GREY, "o"),
    ("pca", BLUE, "s"),
    ("ae_frozen", ORANGE, "^"),
    ("ae_finetuned", GREEN, "D"),
    ("scratch", RED, "v"),
):
    ax.plot(res["per_class"], res[col], color=c, marker=mk, ms=3, label=col.replace("_", " "))
ax.set_xscale("log")
ax.set_xlabel("labelled digits per class")
ax.set_ylabel("test accuracy")
ax.legend(fontsize=6.2, ncol=2, loc="lower right")
fig.set_size_inches(1.76, 1.9)
plotlib.save(fig, "g1_pretrain_curve")

## Slide: Unlabelled Data and the Gain (G1)

Fix 5 labels per class (50 labels) and change how many unlabelled digits are used to pretrain: 0 (random encoder), 100, 400, 1257. Frozen encoder + logistic regression, 3 draws of the labels.

In [ ]:
rows = []
for n_unl in (0, 100, 400, 1257):
    a, e = semisup.autoencoder(64, K, hidden=(32,), seed=0)
    if n_unl:
        semisup.fit_ae(a, Xtr[:n_unl], noise=0.1, epochs=min(400, 150 * 1257 // n_unl))
    Za, Zt = np.asarray(e.predict(Xtr, verbose=0)), np.asarray(e.predict(Xte, verbose=0))
    accs = [lr_acc(Za, Zt, semisup.few_labels(ytr, 5, s)) for s in range(3)]
    rows.append({"unlabelled": n_unl, "acc": float(np.mean(accs)), "std": float(np.std(accs))})
unl = pl.DataFrame(rows)
slide("Unlabelled Data and the Gain")
show(unl)

fig, ax = plotlib.fig("col", 1.5)
ax.errorbar(range(len(unl)), unl["acc"], unl["std"], color=BLUE, marker="o", ms=3, capsize=2)
ax.set_xticks(range(len(unl)), [str(v) for v in unl["unlabelled"]])
ax.set_xlabel("unlabelled digits used for pretraining")
ax.set_ylabel("test accuracy (50 labels)")
plotlib.save(fig, "g1_unlabelled")

## Slide: Mask and Reconstruct: A Pretext Task (G2)

The input has a fraction $p$ of its pixels set to 0 (a new random mask in every epoch); the target is the full digit. The network must use the visible pixels to guess the hidden ones. Frozen encoder + logistic regression on 50 labels (5 per class, 3 draws).

In [ ]:
rows = []
for p in (0.0, 0.25, 0.5, 0.75, 0.9):
    a, e = semisup.autoencoder(64, K, hidden=(32,), seed=0)
    semisup.fit_ae(a, Xtr, mask=p, epochs=150)
    Za, Zt = np.asarray(e.predict(Xtr, verbose=0)), np.asarray(e.predict(Xte, verbose=0))
    accs = [lr_acc(Za, Zt, semisup.few_labels(ytr, 5, s)) for s in range(3)]
    err_masked = float(
        np.mean(
            (
                np.asarray(
                    a.predict((Xte * (np.random.default_rng(1).random(Xte.shape) >= 0.5)).astype("float32"), verbose=0)
                )
                - Xte
            )
            ** 2
        )
    )
    rows.append(
        {"mask_rate": p, "acc": float(np.mean(accs)), "std": float(np.std(accs)), "mse_on_half_masked": err_masked}
    )
msk = pl.DataFrame(rows)
slide("Mask and Reconstruct: A Pretext Task")
show(msk, 4)

fig, ax = plotlib.fig("col", 1.5)
ax.errorbar(msk["mask_rate"], msk["acc"], msk["std"], color=ORANGE, marker="o", ms=3, capsize=2)
ax.set_xlabel("masking rate $p$")
ax.set_ylabel("test accuracy (50 labels)")
plotlib.save(fig, "g2_mask_rate")

## Failure case: a pretext that has nothing to do with the task

Pretrain on 1257 images of uniform noise (reconstructing noise teaches nothing about digits), then the same frozen-encoder probe with 50 labels.

In [ ]:
rng = np.random.default_rng(0)
noise_imgs = rng.random(Xtr.shape).astype("float32")
rows = []
for name, data_ in (("digits", Xtr), ("uniform noise", noise_imgs), ("no pretraining", None)):
    a, e = semisup.autoencoder(64, K, hidden=(32,), seed=0)
    if data_ is not None:
        semisup.fit_ae(a, data_, noise=0.1, epochs=150)
    Za, Zt = np.asarray(e.predict(Xtr, verbose=0)), np.asarray(e.predict(Xte, verbose=0))
    accs = [lr_acc(Za, Zt, semisup.few_labels(ytr, 5, s)) for s in range(3)]
    rows.append({"pretrained on": name, "acc": float(np.mean(accs))})
slide("A pretext that has nothing to do with the task")
show(pl.DataFrame(rows))

## Slide: Which Method for 50 Labels? (G1)

All methods of the class on the same 50 labelled digits (5 per class, 3 draws), test accuracy on the 540 test digits. The graph methods see the test digits as unlabelled nodes (transductive); the others are fitted on the training rows only.
Pretrained codes: the encoder of the first cell (denoising pretext, frozen).

In [ ]:
Xall = np.vstack([Xtr, Xte])
yall = np.concatenate([ytr, yte])
Wg = semisup.knn_affinity(Xall, 10)
rows = {
    "labelled only (pixels)": [],
    "PCA 16 + LR": [],
    "self-training": [],
    "label spreading": [],
    "AE 16 frozen + LR": [],
    "AE 16 fine-tuned": [],
}
for seed in range(3):
    idx = semisup.few_labels(ytr, 5, seed)
    u = np.setdiff1d(np.arange(len(Xtr)), idx)
    rows["labelled only (pixels)"].append(lr_acc(Xtr, Xte, idx))
    rows["PCA 16 + LR"].append(lr_acc(pca.transform(Xtr), pca.transform(Xte), idx))
    m, _ = semisup.self_train(Xtr[idx], ytr[idx], Xtr[u], 0.9, 10)
    rows["self-training"].append(m.score(Xte, yte))
    pred = semisup.label_spreading(Wg, yall, idx, 0.9)
    rows["label spreading"].append(float(np.mean(pred[len(Xtr) :] == yte)))
    rows["AE 16 frozen + LR"].append(lr_acc(Ztr_ae, Zte_ae, idx))
    fm = semisup.classifier_from(enc, seed=seed)
    fm.fit(Xtr[idx], ytr[idx], epochs=100, batch_size=16, verbose=0)
    rows["AE 16 fine-tuned"].append(float(fm.evaluate(Xte, yte, verbose=0)[1]))
summary = pl.DataFrame(
    {
        "method": list(rows),
        "acc": [float(np.mean(v)) for v in rows.values()],
        "std": [float(np.std(v)) for v in rows.values()],
    }
)
slide("Which Method for 50 Labels?")
show(summary)

fig, ax = plotlib.fig("col", 1.8)
ax.barh(summary["method"][::-1], summary["acc"][::-1], xerr=summary["std"][::-1], color=BLUE, capsize=2)
ax.set_xlim(0.6, 1.0)
ax.set_xlabel("test accuracy (50 labels)")
ax.grid(axis="y", visible=False)
plotlib.save(fig, "g1_summary")